In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual dos gráficos
sns.set_theme(style="whitegrid")

# Leitura dos dados brutos
df_orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
df_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
df_customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")

# Cruzamento de dados (Mínimo de 3 tabelas unidas)
df_merged = df_orders.merge(df_items, on="order_id").merge(df_customers, on="customer_id")

# Conversão da coluna de data de string para Datetime (essencial para a Análise 2)
df_merged['order_purchase_timestamp'] = pd.to_datetime(df_merged['order_purchase_timestamp'])

# Configuração visual padrão para todos os gráficos do Seaborn
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Bases carregadas com sucesso! Iniciando análises...\n")


Bases carregadas com sucesso! Iniciando análises...



In [13]:
# Análise 1 (Distribuição): Distribuição de compras por estado 
print("Processando Análise 1: Distribuição por Estado...")

# Tabela de frequência absoluta e percentual
dist_estados = df_merged['customer_state'].value_counts()
dist_estados_pct = df_merged['customer_state'].value_counts(normalize=True) * 100
df_dist_estados = pd.DataFrame({'Total Pedidos': dist_estados, 'Percentual (%)': dist_estados_pct.round(2)})
print("\nTabela de Distribuição por Estado (Top 10):")
print(df_dist_estados.head(10))

# Gráfico de barras da distribuição por estado
plt.figure()
sns.countplot(data=df_merged, x='customer_state', order=dist_estados.index, palette='viridis')
plt.title('Distribuição Absoluta de Compras por Estado (UF) no Brasil', fontsize=14, fontweight='bold')
plt.xlabel('Estado (UF)', fontsize=12)
plt.ylabel('Quantidade de Itens Comprados', fontsize=12)
plt.tight_layout()
plt.savefig('../docs/analise1_distribuicao_estados.png')
plt.close()


Processando Análise 1: Distribuição por Estado...

Tabela de Distribuição por Estado (Top 10):
                Total Pedidos  Percentual (%)
customer_state                               
SP                      47449           42.12
RJ                      14579           12.94
MG                      13129           11.65
RS                       6235            5.53
PR                       5740            5.10
SC                       4176            3.71
BA                       3799            3.37
DF                       2406            2.14
GO                       2333            2.07
ES                       2256            2.00


C:\Users\pauli\AppData\Local\Temp\ipykernel_29476\2433174653.py:13: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df_merged, x='customer_state', order=dist_estados.index, palette='viridis')


In [14]:
# Análise 2 (Tendência): Comportamento temporal de compras utilizando a coluna de data 
# convertendo order_purchase_timestamp para tipo Datetime
print("\nProcessando Análise 2: Tendência Temporal...")

# Criando coluna de Ano-Mês para agrupamento temporal
df_merged['ano_mes'] = df_merged['order_purchase_timestamp'].dt.to_period('M')

# Agrupando volume de pedidos por mês
vendas_mensais = df_merged.groupby('ano_mes').size()

print("\nTabela de Tendência Temporal (Últimos 10 meses registrados):")
print(vendas_mensais.tail(10))

# Gráfico de linha para visualizar a tendência temporal
plt.figure()
vendas_mensais.plot(kind='line', marker='o', color='teal', linewidth=2.5)
plt.title('Evolução Temporal do Volume de Compras (Mensal)', fontsize=14, fontweight='bold')
plt.xlabel('Período (Ano-Mês)', fontsize=12)
plt.ylabel('Quantidade de Pedidos', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../docs/analise2_tendencia_temporal.png')
plt.close()


Processando Análise 2: Tendência Temporal...

Tabela de Tendência Temporal (Últimos 10 meses registrados):
ano_mes
2017-12    6308
2018-01    8208
2018-02    7672
2018-03    8217
2018-04    7975
2018-05    7925
2018-06    7078
2018-07    7092
2018-08    7248
2018-09       1
Freq: M, dtype: int64


In [4]:
# Análise 3 (Dispersão): Correlação e dispersão entre Preço do Item (price) e Valor do Frete (freight_value).
print("\nProcessando Análise 3: Dispersão e Correlação...")

# Calculando o coeficiente de correlação de Pearson
correlacao = df_merged['price'].corr(df_merged['freight_value'])
print(f"\nCoeficiente de Correlação de Pearson entre Preço e Frete: {correlacao:.4f}")

# Gráfico de dispersão com uma amostra dos dados para evitar lentidão (10.000 pontos)
plt.figure()
df_sample = df_merged.sample(10000, random_state=42)
sns.scatterplot(data=df_sample, x='price', y='freight_value', alpha=0.5, color='darkorange')
plt.title(f'Relação entre Preço do Produto e Valor do Frete (Correlação: {correlacao:.2f})', fontsize=14, fontweight='bold')
plt.xlabel('Preço do Produto (R$)', fontsize=12)
plt.ylabel('Valor do Frete (R$)', fontsize=12)
# Limitando eixos para focar na concentração principal dos dados
plt.xlim(0, 1000)
plt.ylim(0, 200)
plt.tight_layout()
plt.savefig('../docs/analise3_dispersao_preco_frete.png')
plt.close()



Processando Análise 3: Dispersão e Correlação...


NameError: name 'df_merged' is not defined

In [6]:
# Análise 4 (Média/Mediana): Ticket médio e frete médio por estado
print("\nProcessando Análise 4: Média e Mediana por Estado...")

# 1. Agrupando por estado e calculando a média e mediana do preço e do frete
df_agrupado = df_merged.groupby('customer_state').agg({
    'price': ['mean', 'median'],
    'freight_value': ['mean', 'median']
}).round(2)

# 2. Corrigindo o MultiIndex: Achatando as colunas para evitar erros de indexação
df_agrupado.columns = [
    'Preco_Medio', 'Preco_Mediana', 
    'Frete_Medio', 'Frete_Mediana'
]

# Resetando o índice para que 'customer_state' volte a ser uma coluna comum
df_agrupado = df_agrupado.reset_index()

# 3. Ordenando os dados de forma correta e segura
df_ordenado = df_agrupado.sort_values(by='Preco_Medio', ascending=False)

print("\nTabela Comparativa de Média e Mediana por Estado (Top 10 por Preço Médio):")
# Renomeando temporariamente apenas para exibição bonita no console
print(df_ordenado.rename(columns={
    'customer_state': 'Estado',
    'Preco_Medio': 'Preço Médio (R$)',
    'Preco_Mediana': 'Preço Mediana (R$)',
    'Frete_Medio': 'Frete Médio (R$)',
    'Frete_Mediana': 'Frete Mediana (R$)'
}).head(10).to_string(index=False))



# GERAÇÃO DO GRÁFICO PARA A ANÁLISE 4

# Vamos selecionar os 10 estados com maiores médias de preço para criar um gráfico comparativo duplo

df_plot = df_ordenado.head(10)

# Criando a estrutura do gráfico lado a lado (Preço Médio vs. Frete Médio)
fig, ax1 = plt.subplots(figsize=(14, 7))

# Configurando o eixo esquerdo para o Preço Médio (Barras Azuis)
color = 'royalblue'
ax1.set_xlabel('Estado (UF)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Preço Médio do Produto (R$)', color=color, fontsize=12, fontweight='bold')
bars = ax1.bar(df_plot['customer_state'], df_plot['Preco_Medio'], color=color, alpha=0.7, label='Preço Médio')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(False) # Desativa grade interna para não poluir

# Adicionando rótulos de valores em cima das barras de preço
for bar in bars:
    height = bar.get_height()
    ax1.annotate(f'R$ {height}',
                 xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 3),  # Deslocamento vertical de 3 pontos
                 textcoords="offset points",
                 ha='center', va='bottom', fontsize=9, fontweight='bold', color=color)

# Criando um segundo eixo Y (direito) compartilhado para o Frete Médio (Linha Vermelha)
ax2 = ax1.twinx()  
color_line = 'crimson'
ax2.set_ylabel('Valor Médio do Frete (R$)', color=color_line, fontsize=12, fontweight='bold')
line = ax2.plot(df_plot['customer_state'], df_plot['Frete_Medio'], color=color_line, marker='o', linewidth=2.5, label='Frete Médio')
ax2.tick_params(axis='y', labelcolor=color_line)
ax2.grid(False)

# Adicionando rótulos de valores nos pontos da linha de frete
for i, txt in enumerate(df_plot['Frete_Medio']):
    ax2.annotate(f'R$ {txt}', 
                 (df_plot['customer_state'].iloc[i], df_plot['Frete_Medio'].iloc[i]),
                 textcoords="offset points", 
                 xytext=(0,10), 
                 ha='center', fontsize=9, fontweight='bold', color=color_line)

plt.title('Comparativo de Preço Médio do Produto vs. Valor Médio do Frete por Estado (Top 10 UFs por Preço)', fontsize=14, fontweight='bold', pad=20)
fig.tight_layout()

# Salvando o gráfico de forma organizada na pasta de documentos do projeto
plt.savefig('../docs/analise4_comparativo_preco_frete.png', dpi=300)
plt.close()

print("\n[Sucesso] Gráfico da Análise 4 gerado e salvo em: '../docs/analise4_comparativo_preco_frete.png'")




Processando Análise 4: Média e Mediana por Estado...

Tabela Comparativa de Média e Mediana por Estado (Top 10 por Preço Médio):
Estado  Preço Médio (R$)  Preço Mediana (R$)  Frete Médio (R$)  Frete Mediana (R$)
    PB            191.48               99.97             42.72               34.29
    AL            180.89               99.94             35.84               33.16
    AC            173.73               99.94             40.07               35.74
    RO            165.97               95.50             41.07               34.70
    PA            165.69               99.00             35.83               27.90
    AP            164.32               99.70             34.01               29.78
    PI            160.36               99.99             39.15               34.30
    TO            157.53               89.49             37.25               29.10
    RN            156.97               98.00             35.65               34.15
    CE            153.76               9

In [17]:
# Análise 5 (Outliers): Identificação de outliers no valor do frete
print("\nProcessando Análise 5: Identificação de Outliers...")

# Identificação matemática de outliers usando a técnica do Intervalo Interquartil (IQR)
q1 = df_merged['freight_value'].quantile(0.25)
q3 = df_merged['freight_value'].quantile(0.75)
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr

outliers_frete = df_merged[df_merged['freight_value'] > limite_superior]
percentual_outliers = (len(outliers_frete) / len(df_merged)) * 100

print(f"\nEstatísticas de Frete:")
print(f"  - 1º Quartil (Q1): R$ {q1:.2f}")
print(f"  - 3º Quartil (Q3): R$ {q3:.2f}")
print(f"  - Limite para ser considerado outlier: R$ {limite_superior:.2f}")
print(f"  - Quantidade de registros outliers: {len(outliers_frete)}")
print(f"  - Representação na base de dados: {percentual_outliers:.2f}% dos dados")

# Gráfico Boxplot para visualizar a distribuição e os outliers do frete
plt.figure()
sns.boxplot(x=df_merged['freight_value'], color='lightblue')
plt.title('Identificação de Outliers no Valor do Frete Cobrado', fontsize=14, fontweight='bold')
plt.xlabel('Valor do Frete (R$)', fontsize=12)
plt.tight_layout()
plt.savefig('../docs/analise5_outliers_frete.png')
plt.close()

print("\nTodas as análises exploratórias foram executadas e os gráficos salvos na pasta 'docs/'!")


Processando Análise 5: Identificação de Outliers...

Estatísticas de Frete:
  - 1º Quartil (Q1): R$ 13.08
  - 3º Quartil (Q3): R$ 21.15
  - Limite para ser considerado outlier: R$ 33.25
  - Quantidade de registros outliers: 11613
  - Representação na base de dados: 10.31% dos dados

Todas as análises exploratórias foram executadas e os gráficos salvos na pasta 'docs/'!
